# Chuẩn bị dữ liệu Artist CDNMF

Notebook này chuyển `artist_csb.csv` và `edges.csv` thành định dạng text mà `Dataset/dataset.py` đọc được, lưu tại `Database/artist_csb/`. Cột `main_department` được dùng làm nhãn đánh giá vì CSV không có nhãn cộng đồng riêng. Chạy từ thư mục gốc repository; sau đó chọn `DATASET = "artist_csb"` trong `run_model.ipynb`.

In [ ]:
from collections import Counter
import json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
ARTIST_CSV = ROOT / "artist_csb.csv"
EDGES_CSV = ROOT / "edges.csv"
OUTPUT_DIR = ROOT / "Database" / "artist_csb"
MAX_CATEGORY_VALUES = 64
MAX_TAG_FEATURES = 256

if not ARTIST_CSV.is_file() or not EDGES_CSV.is_file():
    raise FileNotFoundError("Đặt notebook và hai file CSV tại thư mục gốc CDNMF trước khi chạy.")
if not (ROOT / "README.md").is_file() or not (ROOT / "Dataset" / "dataset.py").is_file():
    raise FileNotFoundError("Hãy chạy notebook từ thư mục gốc của repository CDNMF.")

print(f"Artists: {ARTIST_CSV}\nEdges: {EDGES_CSV}\nOutput: {OUTPUT_DIR}")

## 1. Nạp hai CSV

Giữ ID ở dạng chuỗi để đảm bảo khóa trong file cạnh khớp chính xác với ID nghệ sĩ.

In [ ]:
artists = pd.read_csv(ARTIST_CSV, dtype={"artist_id": "string"})
edges = pd.read_csv(
    EDGES_CSV,
    dtype={"source": "string", "target": "string"},
)

required_artist_columns = {
    "artist_id", "artwork_count", "subject_tags", "style_tags",
    "technique_tags", "material_tags", "main_department",
    "main_century", "main_place_of_origin",
}
required_edge_columns = {"source", "target", "weight"}
missing_artist_columns = required_artist_columns - set(artists.columns)
missing_edge_columns = required_edge_columns - set(edges.columns)
if missing_artist_columns or missing_edge_columns:
    raise ValueError(
        f"Thiếu cột CSV. artist_csb.csv: {sorted(missing_artist_columns)}; "
        f"edges.csv: {sorted(missing_edge_columns)}"
    )

print(f"Đã nạp {len(artists):,} nghệ sĩ và {len(edges):,} cạnh.")

## 2. Xem cấu trúc và giá trị thiếu

Xem nhanh mẫu dữ liệu, kiểu cột và số giá trị thiếu để phát hiện CSV sai schema trước khi tạo file model.

In [ ]:
display(artists.head())
display(edges.head())

print("Artist columns and dtypes:")
display(artists.dtypes.to_frame("dtype"))
print("Missing values in artist CSV:")
display(artists.isna().sum().to_frame("missing"))
print("Missing values in edge CSV:")
display(edges.isna().sum().to_frame("missing"))
print(f"Unique artist IDs: {artists['artist_id'].nunique():,}")
print(f"Unique edge endpoint IDs: {pd.unique(pd.concat([edges['source'], edges['target']])).size:,}")
print(f"Department labels: {artists['main_department'].nunique(dropna=True):,}")

## 3. Tiền xử lý dữ liệu cho CDNMF

Tạo một dòng đặc trưng không âm cho mỗi nghệ sĩ. Các danh mục/tag có giới hạn số chiều; ID cạnh không có metadata sẽ dừng xử lý để tránh lệch ma trận.

In [ ]:
artists = artists.copy()
edges = edges.copy()
artists["artist_id"] = artists["artist_id"].astype("string").str.strip()
edges["source"] = edges["source"].astype("string").str.strip()
edges["target"] = edges["target"].astype("string").str.strip()

if artists["artist_id"].isna().any() or artists["artist_id"].eq("").any():
    raise ValueError("artist_csb.csv có artist_id bị thiếu hoặc rỗng.")
if artists["artist_id"].duplicated().any():
    examples = artists.loc[artists["artist_id"].duplicated(), "artist_id"].head().tolist()
    raise ValueError(f"artist_id bị trùng, ví dụ: {examples}")
if artists["artist_id"].str.contains(r"\s", regex=True).any():
    raise ValueError("artist_id không được chứa khoảng trắng vì định dạng model phân tách bằng space.")
if edges[["source", "target"]].isna().any().any() or edges[["source", "target"]].eq("").any().any():
    raise ValueError("edges.csv có source/target bị thiếu hoặc rỗng.")

edges["weight"] = pd.to_numeric(edges["weight"], errors="coerce")
if edges["weight"].isna().any() or not np.isfinite(edges["weight"]).all():
    raise ValueError("Cột weight phải chứa số hữu hạn.")
if (edges["weight"] < 0).any():
    raise ValueError("CDNMF dùng NMF không âm; weight âm không được hỗ trợ.")

node_ids = sorted(artists["artist_id"].astype(str).tolist())
node_id_set = set(node_ids)
edge_node_ids = set(edges["source"].astype(str)) | set(edges["target"].astype(str))
missing_metadata_ids = sorted(edge_node_ids - node_id_set)
if missing_metadata_ids:
    raise ValueError(
        f"Có {len(missing_metadata_ids)} node trong edges.csv không có artist metadata; "
        f"ví dụ: {missing_metadata_ids[:10]}"
    )

artists = artists.set_index(artists["artist_id"].astype(str), drop=False).loc[node_ids]


def minmax_feature(column):
    values = pd.to_numeric(artists[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
    if values.notna().any():
        values = values.fillna(values.median()).to_numpy(dtype=np.float64)
        low, high = values.min(), values.max()
        return np.zeros(len(values), dtype=np.float32) if high == low else ((values - low) / (high - low)).astype(np.float32)
    return np.zeros(len(artists), dtype=np.float32)


def clean_category(series):
    values = series.fillna("__unknown__").astype(str).str.strip()
    return values.mask(values.eq(""), "__unknown__")


def capped_one_hot(series, prefix, limit):
    values = clean_category(series)
    counts = values.value_counts()
    kept = sorted(counts.index.tolist(), key=lambda value: (-counts[value], value))[:limit]
    has_other = bool((~values.isin(kept)).any())
    names = [f"{prefix}={value}" for value in kept]
    if has_other:
        names.append(f"{prefix}=__other__")
    encoded = np.zeros((len(values), len(names)), dtype=np.float32)
    positions = {value: index for index, value in enumerate(kept)}
    other_index = len(kept) if has_other else None
    for row_index, value in enumerate(values):
        encoded[row_index, positions.get(value, other_index)] = 1.0
    return encoded, names


def parse_tag_dict(value, column, row_index):
    if pd.isna(value) or not str(value).strip():
        return {}
    try:
        parsed = json.loads(value)
    except (TypeError, json.JSONDecodeError) as error:
        raise ValueError(f"JSON tag không hợp lệ ở {column}, dòng {row_index + 2}.") from error
    if not isinstance(parsed, dict):
        raise ValueError(f"{column} phải là JSON object ở dòng {row_index + 2}.")
    return parsed


feature_parts = [minmax_feature("artwork_count"), minmax_feature("main_century")]
feature_names = ["artwork_count_minmax", "main_century_minmax"]
for category_column, category_prefix in (
    ("main_department", "department"),
    ("main_place_of_origin", "origin"),
):
    encoded, names = capped_one_hot(artists[category_column], category_prefix, MAX_CATEGORY_VALUES)
    feature_parts.append(encoded)
    feature_names.extend(names)

tag_columns = ("subject_tags", "style_tags", "technique_tags", "material_tags")
parsed_tags = []
tag_counts = Counter()
for row_index, (_, artist) in enumerate(artists.iterrows()):
    row_tags = {}
    for column in tag_columns:
        for tag, count in parse_tag_dict(artist[column], column, row_index).items():
            token = f"{column.removesuffix('_tags')}={tag}"
            row_tags[token] = max(float(count), 0.0)
    parsed_tags.append(row_tags)
    tag_counts.update(row_tags.keys())

top_tags = sorted(tag_counts, key=lambda tag: (-tag_counts[tag], tag))[:MAX_TAG_FEATURES]
tag_positions = {tag: index for index, tag in enumerate(top_tags)}
tag_features = np.zeros((len(node_ids), len(top_tags)), dtype=np.float32)
for row_index, row_tags in enumerate(parsed_tags):
    for tag in row_tags:
        if tag in tag_positions:
            tag_features[row_index, tag_positions[tag]] = 1.0
feature_parts.append(tag_features)
feature_names.extend(f"tag:{tag}" for tag in top_tags)

X = np.column_stack(feature_parts).astype(np.float32, copy=False)
if X.shape[0] != len(node_ids) or X.shape[1] == 0:
    raise ValueError(f"Ma trận feature sai kích thước: {X.shape}")
if not np.isfinite(X).all() or (X < 0).any():
    raise ValueError("Ma trận feature phải chứa số hữu hạn không âm.")

labels = clean_category(artists["main_department"])
label_values = sorted(labels.unique().tolist())
label_to_id = {label: f"department_{index:03d}" for index, label in enumerate(label_values)}
label_ids = labels.map(label_to_id).to_numpy()
node_to_row = {node_id: row_index for row_index, node_id in enumerate(node_ids)}

print(f"Prepared {len(node_ids):,} nodes, {len(edges):,} edges, {X.shape[1]:,} features, {len(label_values):,} labels.")

## 4. Lưu dữ liệu cho `run_model.ipynb`

File được ghi theo đúng thứ tự node và định dạng cách nhau bằng dấu cách mà `Dataset` hiện tại yêu cầu. `walks.txt` để rỗng vì model chỉ nạp file này, chưa dùng trong loss.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
feature_frame = pd.DataFrame(X, index=node_ids)
feature_frame.to_csv(
    OUTPUT_DIR / "features.txt",
    sep=" ",
    header=False,
    index=True,
    index_label=False,
    float_format="%.7g",
    lineterminator="\n",
)
edges[["source", "target", "weight"]].to_csv(
    OUTPUT_DIR / "edges.txt",
    sep=" ",
    header=False,
    index=False,
    float_format="%.8g",
    lineterminator="\n",
)
pd.DataFrame({"node_id": node_ids, "label": label_ids}).to_csv(
    OUTPUT_DIR / "group.txt",
    sep=" ",
    header=False,
    index=False,
    lineterminator="\n",
)
pd.DataFrame(
    [(safe_id, label) for label, safe_id in label_to_id.items()],
    columns=["label_id", "main_department"],
).to_csv(OUTPUT_DIR / "label_mapping.csv", index=False, encoding="utf-8")
(OUTPUT_DIR / "walks.txt").write_text("", encoding="utf-8")

minimum_dense_bytes = 2 * len(node_ids) ** 2 * np.dtype(np.float64).itemsize
print(f"Đã lưu dữ liệu tại: {OUTPUT_DIR}")
print(f"Kích thước features: {len(node_ids):,} x {X.shape[1]:,}")
print(f"Nhãn: {len(label_to_id)}; mapping: {OUTPUT_DIR / 'label_mapping.csv'}")
print(
    "Lưu ý bộ nạp hiện tại dùng ma trận dense: riêng A và W cần ít nhất "
    f"{minimum_dense_bytes / (1024 ** 2):.1f} MiB RAM; huấn luyện cần thêm RAM/VRAM."
)

## 5. Kiểm tra file đầu ra

Đọc kiểm tra định dạng và tính liên kết ID mà không khởi tạo ma trận dense của `Dataset`.

In [ ]:
feature_check = pd.read_csv(OUTPUT_DIR / "features.txt", sep=r"\s+", header=None, dtype={0: str})
edge_check = pd.read_csv(
    OUTPUT_DIR / "edges.txt",
    sep=r"\s+",
    header=None,
    names=["source", "target", "weight"],
    dtype={"source": str, "target": str},
)
group_check = pd.read_csv(
    OUTPUT_DIR / "group.txt",
    sep=r"\s+",
    header=None,
    names=["node_id", "label"],
    dtype=str,
)

assert len(feature_check) == len(node_ids) == len(group_check)
assert feature_check.iloc[:, 0].tolist() == node_ids
assert group_check["node_id"].tolist() == node_ids
assert feature_check.shape[1] - 1 == len(feature_names)
assert np.isfinite(feature_check.iloc[:, 1:].to_numpy(dtype=np.float32)).all()
assert (feature_check.iloc[:, 1:].to_numpy(dtype=np.float32) >= 0).all()
assert set(edge_check["source"]) | set(edge_check["target"]) <= set(node_ids)
assert np.isfinite(edge_check["weight"]).all() and (edge_check["weight"] >= 0).all()
assert len((OUTPUT_DIR / "walks.txt").read_text(encoding="utf-8")) == 0

print(
    f"OK: {len(feature_check):,} feature rows x {feature_check.shape[1] - 1:,} features; "
    f"{len(edge_check):,} edges; {group_check['label'].nunique():,} labels."
)